# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pdimport numpy as npimport matplotlib.pyplot as plt# Define the Croissant schema JSON-LD URLcroissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'# Load the dataset metadatadataset = mlc.Dataset(croissant_url)# Access metadata attributes without subscripting as a dictionaryds_metadata = dataset.metadataprint(f"{ds_metadata.name}: {ds_metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets and their fields by @idrecord_sets = dataset.metadata.record_setsif not record_sets:    print('No record sets defined in this Croissant schema.')else:    for rs in record_sets:        print(f"RecordSet: @{rs['@id']} -- name: {rs['name']}  description: {rs.get('description','')}")        print("  Fields:")        for field in rs.get('fields', []):            print(f"    Field: @{field['@id']} -- name: {field['name']}  datatype: {field.get('data_type','')} source_column @id: {field.get('source_column','')}")        print()# For convenience, print all available record set @id'sprint('Available RecordSet @id\'s:')if record_sets:    for rs in record_sets:        print(f"- {rs['@id']}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Example: Load all records from each record set into a pandas DataFrame
# (You may need to adjust the chosen RecordSet @id based on the output from Section 2)
# In case the dataset defines record sets, let's extract their ids:record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets] if dataset.metadata.record_sets else []dataframes = {}for rs_id in record_set_ids:    # Load all records for this record set    records = list(dataset.records(record_set=rs_id))    df = pd.DataFrame(records)    dataframes[rs_id] = df    print(f"Loaded {len(df)} records from RecordSet @{rs_id}.")if record_set_ids:    print(f"Sample columns in @{record_set_ids[0]}:")    print(dataframes[record_set_ids[0]].columns.tolist())    display(dataframes[record_set_ids[0]].head())else:    print('No record sets found. The record structure may be directly in the distribution files or need schema inspection.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

_**Note**_: If fields or record sets were not found, you may need to update the field or record set IDs based on the listing above.

In [ ]:
# EDA Example: Assume a record set and numeric field are availableif record_set_ids:    # For demonstration, choose the first record set    eda_rs_id = record_set_ids[0]    df = dataframes[eda_rs_id]    print(f'Available columns in @{eda_rs_id}:')    print(df.columns.tolist())    # Try to automatically select a numeric field    # Or set manually if known, e.g. numeric_field = '@PERCENT_COVERAGE' (edit this as per real field IDs)    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()    if not numeric_fields:        # Try to coerce columns that might be numeric        for col in df.columns:            try:                df[col] = pd.to_numeric(df[col])            except Exception: pass        numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()    if numeric_fields:        numeric_field = numeric_fields[0]        print(f"Using numeric field '@{numeric_field}' for EDA.")        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 10        filtered_df = df[df[numeric_field] > threshold]        print(f"Filtered records where @{numeric_field} > {threshold:.2f} (mean):")        print(filtered_df.head(), '\n')        # Normalize        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()        print("First 5 normalized values:")        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())        # Try grouping by a candidate field (pick next available non-numeric field, e.g. protein name or id)        group_fields = [c for c in df.columns if c != numeric_field]        group_field = None        for gf in group_fields:            if df[gf].dtype == object:                group_field = gf                break        if group_field:            print(f"Grouping by @{group_field} (example):")            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()            print(grouped_df.head())    else:        print('No numeric field automatically detected. Please inspect the columns above.')else:    print('No available record set for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Histogram or scatterplot for the selected numeric field (if available)if record_set_ids and numeric_fields:    plt.figure(figsize=(7,4))    df = dataframes[eda_rs_id]    plt.hist(df[numeric_field].dropna(), bins=30, color='skyblue', edgecolor='black')    plt.title(f'Distribution of @{numeric_field}')    plt.xlabel(numeric_field)    plt.ylabel('Count')    plt.show()    # If grouping field identified, scatter mean values    if group_field and 'grouped_df' in locals():        plt.figure(figsize=(8,4))        plt.bar(grouped_df[group_field].astype(str), grouped_df[numeric_field])        plt.title(f'Mean of @{numeric_field} by @{group_field}')        plt.xlabel(group_field)        plt.ylabel(f'Mean {numeric_field}')        plt.xticks(rotation=30)        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to access and explore the FAIR^2 dataset on protein analysis using `mlcroissant`. We loaded metadata, listed available record sets and fields (referenced consistently by their `@id` fields), extracted records, ran basic EDA and normalization, and visualized distributions.

**Key steps:**
- All dataset schema elements (record sets, fields, columns) were referenced by their `@id` property.
- Dataframes were created for each record set and basic exploration was performed on numeric fields.
- The `mlcroissant` library easily unifies data and metadata extraction for FAIR datasets described by Croissant schemas.

For more advanced use, consult the [mlcroissant documentation](https://mlco.readthedocs.io/) and the dataset's own metadata documentation.